Start by loading all the clip data from _clip_data.jsonl 

In [2]:
import os
import tools.clip_data as cd

clipdata = cd.clip_data_from_jsonl('../ungitable/_clip_data.jsonl')


In [6]:
import json

# Define the output file path
output_file_path = '../ungitable/clipdata_output.json'

# Convert clipdata to a list of dictionaries
clipdata_dict = [clip.model_dump() for clip in clipdata]

#convert fileDownloadUrl to a string
for clip in clipdata_dict:
    clip['fileDownloadUrl'] = str(clip['fileDownloadUrl'])

# Write the data to a JSON file
with open(output_file_path, 'w') as f:
    json.dump(clipdata_dict, f, indent=4)

In [23]:
import pandas as pd

# Extract the required data
data = []
for clip in clipdata:
    event = clip.events[0] if clip.events else None
    if event and event.type == "contact":
        data.append(
            {
                "id": clip.clipId,
                "cur_desc": clip.textDescriptionBrief,
                "cur_predict_ts": event.ts,
            }
        )
    else:
        data.append(
            {
                "id": clip.clipId,
                "cur_desc": clip.textDescriptionBrief,
                "cur_predict_ts": 0.0,
            }
        )

# Create the DataFrame
df = pd.DataFrame(data)
# print(df)

In [24]:
from tools.object_track_types import DeepsortOutput, Annotation

def parse_files(deepsort_file_path, annotation_file_path)->tuple[DeepsortOutput, Annotation]:
    deepsort_data: DeepsortOutput
    annotation_data: Annotation
    with open(deepsort_file_path, 'r') as f:
        deepsort_data = DeepsortOutput.model_validate_json(f.read())
    
    with open(annotation_file_path, 'r') as f:
        annotation_data = Annotation.model_validate_json(f.read()) 

    return deepsort_data, annotation_data


In [25]:


# Add placeholder columns to the DataFrame
from tools.find_paths import build_clip_prediction, build_tracking_data


df['actual_est_ts'] = None
df['predicted_ts'] = None

# Directory containing the files
video_dir = '../ungitable/video'

# Iterate through the clipdata
for clip in clipdata:
    clip_id = clip.clipId
    deepsort_file = os.path.join(video_dir, f"{clip_id}.mp4.deepsort.json")
    annotation_file = os.path.join(video_dir, f"{clip_id}.annotation.json")
    
    # Check if both files exist
    if os.path.isfile(deepsort_file) and os.path.isfile(annotation_file):
        print(f"Processing {clip_id}")
        # Update the DataFrame with placeholder values
        tracking_data = build_tracking_data(
            deepsort_file=deepsort_file,
            annotation_file=annotation_file,
            home_radius=150,
            auto_home_radius=True,
            first_radius=35,
            frame_width=1280,
            frame_height=720,
        )
        clip_prediction = build_clip_prediction(tracking_data=tracking_data, fps=30)


        df.loc[df['id'] == clip_id, 'actual_est_ts'] = tracking_data.annotation.contact_time or 0.0
        df.loc[df['id'] == clip_id, 'predicted_ts'] = clip_prediction.contact_moment
        df.loc[df['id'] == clip_id, 'predicted_event_type'] = clip_prediction.event_type
    # else:
    #     print(f"Both files not found for {clip_id}")

# Print rows with non-null values in 'actual_est_ts' and 'predicted_ts'
# print(len(df.dropna(subset=['actual_est_ts', 'predicted_ts'])))


Processing 20rklLb793TvVuzD
Auto home tolerance: 41.968823879446624
Processing 7qHyeBUcNuXt7KMR
Auto home tolerance: 39.136653183120124
Processing D1b2ZrQuEdo6mLEl
Auto home tolerance: 95.18249336564298
Processing M8E6uaAujZkL0Lyc
Auto home tolerance: 71.0888888135952
Processing 382LnPm2cD9NMt3P
Auto home tolerance: 66.99875914706361
Processing 2hVkAP3woJGeix9e
Auto home tolerance: 102.61735718624672
Processing QQC1wZtPJtaC0cy2
Auto home tolerance: 105.5115902048687
Processing 2cKVta2pHP2Eq2ts
Auto home tolerance: 70.57518033994423
Processing 76IjK5NR3tlO1087
Auto home tolerance: 139.62678777435931
Processing G8DYv6aSExcZbF5l
Auto home tolerance: 154.83593202999245
Processing 3jcTkCX4Sew298p3
Auto home tolerance: 87.15178364455242
Processing 6cCuJiEJBNTx8bHz
Auto home tolerance: 99.6779741717251
Processing 2O7RshsveaiZGfdX
Auto home tolerance: 134.33370923249055
Processing 6vUELxZ6Js4lmfhB
Auto home tolerance: 86.81934519202882
Processing 3VDScNtAwzmHY4qT
Auto home tolerance: 115.00854

In [26]:
df_filtered = df.dropna(subset=['actual_est_ts'])
df_filtered

,id,cur_desc,cur_predict_ts,actual_est_ts,predicted_ts,predicted_event_type
0,20rklLb793TvVuzD,single,24.928,25.866656,26.866667,hit
1,7qHyeBUcNuXt7KMR,out,24.320,27.006663,3.533333,no-hit
2,D1b2ZrQuEdo6mLEl,triple,24.336,25.02,26.066667,hit
3,M8E6uaAujZkL0Lyc,out,0.000,17.733326,18.866667,no-hit
4,382LnPm2cD9NMt3P,single,26.240,26.626666,28.033333,hit
5,2hVkAP3woJGeix9e,single,15.696,15.84,15.833333,hit
6,QQC1wZtPJtaC0cy2,single,15.104,15.599994,16.766667,hit
7,2cKVta2pHP2Eq2ts,double,15.616,16.129994,17.633333,hit
8,76IjK5NR3tlO1087,single,28.192,11.106666,3.266667,no-hit
9,G8DYv6aSExcZbF5l,flyout,22.576,22.933324,3.2,no-hit


In [27]:
df_filtered['cur_correct'] = abs(df_filtered['cur_predict_ts'] - df_filtered['actual_est_ts']) <= 2
df_filtered['pred_correct'] = abs(df_filtered['predicted_ts'] - df_filtered['actual_est_ts']) <= 2
df_filtered

/tmp/ipykernel_75003/1787497557.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['cur_correct'] = abs(df_filtered['cur_predict_ts'] - df_filtered['actual_est_ts']) <= 2
/tmp/ipykernel_75003/1787497557.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['pred_correct'] = abs(df_filtered['predicted_ts'] - df_filtered['actual_est_ts']) <= 2


,id,cur_desc,cur_predict_ts,actual_est_ts,predicted_ts,predicted_event_type,cur_correct,pred_correct
0,20rklLb793TvVuzD,single,24.928,25.866656,26.866667,hit,True,True
1,7qHyeBUcNuXt7KMR,out,24.320,27.006663,3.533333,no-hit,False,False
2,D1b2ZrQuEdo6mLEl,triple,24.336,25.02,26.066667,hit,True,True
3,M8E6uaAujZkL0Lyc,out,0.000,17.733326,18.866667,no-hit,False,True
4,382LnPm2cD9NMt3P,single,26.240,26.626666,28.033333,hit,True,True
5,2hVkAP3woJGeix9e,single,15.696,15.84,15.833333,hit,True,True
6,QQC1wZtPJtaC0cy2,single,15.104,15.599994,16.766667,hit,True,True
7,2cKVta2pHP2Eq2ts,double,15.616,16.129994,17.633333,hit,True,True
8,76IjK5NR3tlO1087,single,28.192,11.106666,3.266667,no-hit,False,False
9,G8DYv6aSExcZbF5l,flyout,22.576,22.933324,3.2,no-hit,True,False


In [28]:
cur_correct_percent = df_filtered['cur_correct'].mean() * 100
pred_correct_percent = df_filtered['pred_correct'].mean() * 100

cur_correct_percent, pred_correct_percent

(np.float64(71.42857142857143), np.float64(71.42857142857143))

In [29]:
grouped = df_filtered.groupby('predicted_event_type').agg(
    cur_correct_percent=('cur_correct', 'mean'),
    pred_correct_percent=('pred_correct', 'mean')
)

# Convert to percentage
grouped['cur_correct_percent'] *= 100
grouped['pred_correct_percent'] *= 100

grouped

,cur_correct_percent,pred_correct_percent
predicted_event_type,,
hit,83.333333,100.000000
no-hit,44.444444,22.222222
unknown,100.000000,0.000000


In [30]:
df_filtered_sorted = df_filtered.sort_values(by='id')
df_filtered_sorted

,id,cur_desc,cur_predict_ts,actual_est_ts,predicted_ts,predicted_event_type,cur_correct,pred_correct
18,18d4IGOE3bHpdncV,double,8.064,19.47333,20.2,hit,False,True
24,1ZpRVioBxb7EM5jC,single,22.464,20.23333,7.166667,no-hit,False,False
0,20rklLb793TvVuzD,single,24.928,25.866656,26.866667,hit,True,True
27,2Keb4OOS9Edo9Q0a,double,10.832,11.313332,10.6,no-hit,True,True
12,2O7RshsveaiZGfdX,strikeout,0.000,0.0,9.766667,no-hit,True,False
7,2cKVta2pHP2Eq2ts,double,15.616,16.129994,17.633333,hit,True,True
5,2hVkAP3woJGeix9e,single,15.696,15.84,15.833333,hit,True,True
4,382LnPm2cD9NMt3P,single,26.240,26.626666,28.033333,hit,True,True
16,3SY8Kwi01HN5p7K8,single,5.136,16.799989,17.966667,hit,False,True
14,3VDScNtAwzmHY4qT,home run,9.824,10.233331,12.033333,hit,True,True


In [13]:
# Create a new column 'contains_out' that indicates whether 'cur_desc' contains 'out'
df_filtered_sorted['contains_out'] = df_filtered_sorted['cur_desc'].str.contains('out')

# Group by the new column 'contains_out'
grouped_by_out = df_filtered_sorted.groupby('contains_out').agg(
    cur_correct_percent=('cur_correct', 'mean'),
    pred_correct_percent=('pred_correct', 'mean')
)

# Convert to percentage
grouped_by_out['cur_correct_percent'] *= 100
grouped_by_out['pred_correct_percent'] *= 100

grouped_by_out

,cur_correct_percent,pred_correct_percent
contains_out,,
False,76.190476,80.952381
True,57.142857,42.857143
